# 阻抗匹配

## 简介下面的图表说明了通用问题：一个内部阻抗为 $Z_S$ 的信号源，通过一个 2 端口匹配网络，向一个无源负载 $Z_L$ 输送功率。这个问题通常被称为“双重匹配问题”。阻抗匹配对于以下原因至关重要：- 最大化功率传输。当信号源和负载都与传输线匹配时，可以向负载输送最大功率，并且可以最大限度地减少传输线中的功率损耗。- 提高系统的信噪比。- 减少幅度和相位误差。- 减少反射回信号源的功率。<img src="figures/Impedance_matching_general.svg">只要负载阻抗 $Z_L$ 具有实数正数部分，就可以始终找到一个匹配网络。有很多选择，下面的示例仅描述了其中的几个。这些示例摘自 D. Pozar 的《微波工程》一书，第四版。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

rf.stylely()

## 使用集中元件进行匹配首先，我们假设匹配网络是无损耗的，并且馈线特性阻抗为 $Z_0$：<img src="figures/Impedance_matching_lumped1.svg">最简单的匹配网络类型是“L”型网络，它使用两个无源元件来匹配任意负载阻抗。存在两种可能的配置，如图所示。在任何一种配置中，无源元件可以是电感或电容，具体取决于负载阻抗。<img src="figures/Impedance_matching_lumped2.svg"><img src="figures/Impedance_matching_lumped3.svg">

假设负载的阻抗为 $Z_L = 200 - 100j \Omega$，传输线的特性阻抗为 $Z_0=100\Omega$，频率为 500 MHz。

In [ ]:
Z_L = 200 - 100j
Z_0 = 100
f_0_str = '500MHz'

让我们定义“频率”和“负载”网络：

In [ ]:
# frequency band centered on the frequency of interest
frequency = rf.Frequency(start=300, stop=700, npoints=401, unit='MHz')
# transmission line Media
line = rf.media.DefinedGammaZ0(frequency=frequency, z0=Z_0)
# load Network
load = line.load(rf.tlineFunctions.zl_2_Gamma0(Z_0, Z_L))

我们正在寻找与上述第一个配置对应的 L-C 网络：<img src="figures/Impedance_matching_lumped4.svg">

In [ ]:
def matching_network_LC_1(L, C):
    ' L and C in nH and pF'
    return line.inductor(L*1e-9)**line.shunt_capacitor(C*1e-12)**load

def matching_network_LC_2(L, C):
    ' L and C in nH and pF'
    return line.capacitor(C*1e-12)**line.shunt_inductor(L*1e-9)**load

寻找合适的电感 $L$ 和电容 $C$ 值以匹配负载是一个优化问题。`scipy` 软件包为此提供了必要的优化函数：

In [ ]:
from scipy.optimize import minimize

# initial guess values
L0 = 10 # nH
C0 = 1 # pF
x0 = (L0, C0)
# bounds
L_minmax = (1, 100) #nH
C_minmax = (0.1, 10) # pF

# the objective functions minimize the return loss at the target frequency f_0
def optim_fun_1(x, f0=f_0_str):
    _ntw = matching_network_LC_1(*x)
    return np.abs(_ntw[f_0_str].s).ravel()

def optim_fun_2(x, f0=f_0_str):
    _ntw = matching_network_LC_2(*x)
    return np.abs(_ntw[f_0_str].s).ravel()

In [ ]:
res1 = minimize(optim_fun_1, x0, bounds=(L_minmax, C_minmax))
print(f'Optimum found for LC network 1: L={res1.x[0]} nH and C={res1.x[1]} pF')

In [ ]:
res2 = minimize(optim_fun_2, x0, bounds=(L_minmax, C_minmax))
print(f'Optimum found for LC network 2: L={res2.x[0]} nH and C={res2.x[1]} pF')

In [ ]:
ntw1 = matching_network_LC_1(*res1.x)
ntw2 = matching_network_LC_2(*res2.x)

In [ ]:
ntw1.plot_s_mag(lw=2, label='LC network 1')
ntw2.plot_s_mag(lw=2, label='LC network 2')
plt.ylim(bottom=0)

## 单端短截匹配可以使用一段开路或短路的传输线（短截）进行匹配，可以并联（分路）或串联连接。在下面的示例中，匹配网络由一段长度为 ($\theta_{stub}$) 的短路传输线并联连接，并与一段串联传输线 ($\theta_{line}$) 关联。假设将一个负载阻抗 $Z_L=60 - 80j$ 连接到 50 欧姆传输线上。<img src="figures/Impedance_matching_stub1.svg">现在，让我们在 2 GHz 频率下匹配这个负载：

In [ ]:
Z_L = 60 - 80j
Z_0 = 50
f_0_str = '2GHz'
# Frequency, wavenumber and transmission line media
freq = rf.Frequency(start=1, stop=3, npoints=301, unit='GHz')
beta = freq.w/rf.constants.c
line = rf.media.DefinedGammaZ0(freq, gamma=1j*beta, z0=Z_0)

In [ ]:
def resulting_network(theta_delay, theta_stub):
    """
    Return a loaded single stub matching network

    NB: theta_delay and theta_stub lengths are in deg
    """
    delay_load = line.delay_load(rf.tlineFunctions.zl_2_Gamma0(Z_0, Z_L), theta_delay)
    shunted_stub = line.shunt_delay_short(theta_stub)
    return shunted_stub ** delay_load

优化匹配网络的变量 `theta_delay` 和 `theta_stub`，以匹配最终的 1 端口网络（$|S|=0$）。

In [ ]:
from scipy.optimize import minimize


def optim_fun(x):
    return resulting_network(*x)[f_0_str].s_mag.ravel()

x0 = (50, 50)
bnd = (0, 180)
res = minimize(optim_fun, x0, bounds=(bnd, bnd))
print(f'Optimum found for: theta_delay={res.x[0]:.1f} deg and theta_stub={res.x[1]:.1f} deg')

In [ ]:
# Optimized network at f0
ntw = resulting_network(*res.x)
ntw.plot_s_db(lw=2)